# Downloading the data from UC Irvine repository

In [1]:
from pathlib import Path

import pandas as pd
from ucimlrepo import fetch_ucirepo

# fetch dataset
iranian_churn = fetch_ucirepo(id=563)

df = pd.concat([iranian_churn.data.features, iranian_churn.data.targets], axis=1)
df.head()

,Call Failure,Complains,Subscription Length,Charge Amount,Seconds of Use,Frequency of use,Frequency of SMS,Distinct Called Numbers,Age Group,Tariff Plan,Status,Age,Customer Value,Churn
0,8,0,38,0,4370,71,5,17,3,1,1,30,197.640,0
1,0,0,39,0,318,5,7,4,2,1,2,25,46.035,0
2,10,0,37,0,2453,60,359,24,3,1,1,30,1536.520,0
3,10,0,38,0,4198,66,1,35,1,1,1,15,240.020,0
4,3,0,38,0,2393,58,2,33,1,1,1,15,145.805,0


In [3]:
data_dir = Path("C:/Users/amade/MLOps_amadeov3/data/iranian_churn")
df.to_parquet(data_dir / "iranian_churn.parquet")

# Exploratory Data Analysis

In [1]:
from pathlib import Path

import pandas as pd

data_dir = Path("C:/Users/amade/MLOps_amadeov3/data/iranian_churn")
df = pd.read_parquet(data_dir / "iranian_churn.parquet")
df.columns = df.columns.str.lower().str.replace({"  ": "_", " ": "_"})

In [2]:
df.describe()

,call_failure,complains,subscription_length,charge_amount,seconds_of_use,frequency_of_use,frequency_of_sms,distinct_called_numbers,age_group,tariff_plan,status,age,customer_value,churn
count,3150.000000,3150.000000,3150.000000,3150.000000,3150.000000,3150.000000,3150.000000,3150.000000,3150.000000,3150.000000,3150.000000,3150.000000,3150.000000,3150.000000
mean,7.627937,0.076508,32.541905,0.942857,4472.459683,69.460635,73.174921,23.509841,2.826032,1.077778,1.248254,30.998413,470.972916,0.157143
std,7.263886,0.265851,8.573482,1.521072,4197.908687,57.413308,112.237560,17.217337,0.892555,0.267864,0.432069,8.831095,517.015433,0.363993
min,0.000000,0.000000,3.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000,15.000000,0.000000,0.000000
25%,1.000000,0.000000,30.000000,0.000000,1391.250000,27.000000,6.000000,10.000000,2.000000,1.000000,1.000000,25.000000,113.801250,0.000000
50%,6.000000,0.000000,35.000000,0.000000,2990.000000,54.000000,21.000000,21.000000,3.000000,1.000000,1.000000,30.000000,228.480000,0.000000
75%,12.000000,0.000000,38.000000,1.000000,6478.250000,95.000000,87.000000,34.000000,3.000000,1.000000,1.000000,30.000000,788.388750,0.000000
max,36.000000,1.000000,47.000000,10.000000,17090.000000,255.000000,522.000000,97.000000,5.000000,2.000000,2.000000,55.000000,2165.280000,1.000000


In [3]:
df.groupby(["churn"]).mean()

,call_failure,complains,subscription_length,charge_amount,seconds_of_use,frequency_of_use,frequency_of_sms,distinct_called_numbers,age_group,tariff_plan,status,age,customer_value
churn,,,,,,,,,,,,,
0,7.656121,0.015443,32.662524,1.075706,5014.224105,76.979284,83.871563,25.582674,2.831638,1.090019,1.155179,31.065913,535.511501
1,7.476768,0.404040,31.894949,0.230303,1566.632323,29.133333,15.802020,12.391919,2.795960,1.012121,1.747475,30.636364,124.811414


In [4]:
df.groupby(["churn"]).median()

,call_failure,complains,subscription_length,charge_amount,seconds_of_use,frequency_of_use,frequency_of_sms,distinct_called_numbers,age_group,tariff_plan,status,age,customer_value
churn,,,,,,,,,,,,,
0,6.0,0.0,35.0,0.0,3530.0,63.0,25.0,23.0,3.0,1.0,1.0,30.0,268.07
1,5.0,0.0,35.0,0.0,1182.0,25.0,9.0,10.0,3.0,1.0,2.0,30.0,96.84


In [6]:
# Let's drop some of the irrelevant/repetitive columns
df = df.drop(columns=["frequency_of_use", "age_group"])

In [7]:
df["churn"].value_counts()

churn
0    2655
1     495
Name: count, dtype: int64

## Backtesting

In [8]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split

RANDOM_SEED = 42
TEST_SIZE = 0.33

In [ ]:
def get_metrics(y_true, y_pred, probs):
    """Compute binary classification metrics.

    Args:
        y_true (pd.Series or np.ndarray): True binary labels.
        y_pred (np.ndarray): Predicted binary labels.
        probs (np.ndarray): Predicted probabilities for class 1.

    Returns:
        dict[str, float]: Accuracy, precision, recall, and ROC-AUC scores.
    """
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    auc = roc_auc_score(y_true, probs)

    return {"accuracy": accuracy, "precision": precision, "recall": recall, "auc": auc}

In [ ]:
def run_experiment(df, model, random_seed, test_size):
    """Fit a classifier on a stratified split and evaluate both subsets.

    Args:
        df (pd.DataFrame): Features and binary target column 'churn'.
        model: Classifier or pipeline supporting fit, predict, and predict_proba.
        random_seed (int): Random seed for splitting the data.
        test_size (float or int): Holdout fraction or number of samples.

    Returns:
        tuple[dict, dict]: Training and holdout metrics, respectively.

    Note:
        Fits the supplied model in place.
    """
    X, y = df.drop(columns=["churn"]), df["churn"]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=random_seed, stratify=y)

    model.fit(X_train, y_train)

    # Prediction on train data:
    y_pred_train = model.predict(X_train)
    probs_train = model.predict_proba(X_train)[:, 1]

    # Prediction on test data:
    y_pred = model.predict(X_test)
    probs_test = model.predict_proba(X_test)[:, 1]

    metrics_train = get_metrics(y_train, y_pred_train, probs_train)
    metrics_test = get_metrics(y_test, y_pred, probs_test)

    return metrics_train, metrics_test

In [11]:
import mlflow
import mlflow.sklearn

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("iranian-churn-experiment")

2026/10/19 13:58:34 INFO mlflow.tracking.fluent: Experiment with name 'iranian-churn-experiment' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/2', creation_time=1792411114653, effective_trace_archival_retention=None, experiment_id='2', last_update_time=1792411114653, lifecycle_stage='active', name='iranian-churn-experiment', tags={}, trace_location=None, workspace='default'>

In [ ]:
def track_experiment(df, model, run_name, random_seed=RANDOM_SEED, test_size=TEST_SIZE):
    """Train and evaluate a classifier, logging results and model to MLflow.

    Args:
        df (pd.DataFrame): Features and binary target column 'churn'.
        model: Classifier or pipeline supporting sklearn-compatible methods.
        run_name (str): Name of the MLflow run and logged model.
        random_seed (int, optional): Split seed. Defaults to RANDOM_SEED.
        test_size (float or int, optional): Holdout size. Defaults to TEST_SIZE.

    Returns:
        dict: Run ID, model URI, training metrics, and validation metrics.
    """
    with mlflow.start_run(run_name=run_name) as run:
        # Log estimator and pipeline parameters.
        # Strings also accommodate settings such as hidden_layer_sizes=(64, 32).
        params = {
            key: str(value)
            for key, value in model.get_params(deep=True).items()
            if key != "steps" and not hasattr(value, "get_params")
        }

        mlflow.log_params(params)
        mlflow.log_params(
            {"split_seed": random_seed, "test_size": test_size, "n_rows": len(df), "n_features": df.shape[1] - 1}
        )

        metrics_train, metrics_valid = run_experiment(df, model, random_seed, test_size)

        for split, metrics in [("train", metrics_train), ("valid", metrics_valid)]:
            mlflow.log_metrics({f"{split}_{name}": float(value) for name, value in metrics.items()})

        model_info = mlflow.sklearn.log_model(model, name=run_name, serialization_format="cloudpickle")

        return {
            "run_id": run.info.run_id,
            "model_uri": model_info.model_uri,
            "train_metrics": metrics_train,
            "valid_metrics": metrics_valid,
        }

### Logistic Regression not Normalised

In [13]:
from sklearn.linear_model import LogisticRegression

logreg = LogisticRegression(C=1.0, l1_ratio=0.0, fit_intercept=True, max_iter=10000)
track_experiment(df, logreg, "logistic_regression")

🏃 View run logistic_regression at: http://127.0.0.1:5000/#/experiments/2/runs/97aa27f79b3f41c083caab65ee5f74ad
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


{'run_id': '97aa27f79b3f41c083caab65ee5f74ad',
 'model_uri': 'models:/m-7d1a8e6daa8a4c9cae884a1771ba9d3d',
 'train_metrics': {'accuracy': 0.8876777251184834,
  'precision': 0.7386934673366834,
  'recall': 0.4427710843373494,
  'auc': 0.9344362828140458},
 'valid_metrics': {'accuracy': 0.8875,
  'precision': 0.7395833333333334,
  'recall': 0.43558282208588955,
  'auc': 0.9235577225762674}}

### Logistic Regression Normalised

In [14]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import MinMaxScaler

logreg_norm = make_pipeline(MinMaxScaler(), LogisticRegression(C=1.0, l1_ratio=0.0, fit_intercept=True, max_iter=10000))
track_experiment(df, logreg_norm, "logistic_regression_normalised")

🏃 View run logistic_regression_normalised at: http://127.0.0.1:5000/#/experiments/2/runs/a78a373fa03a4afe91b8ee035c9d4201
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


{'run_id': 'a78a373fa03a4afe91b8ee035c9d4201',
 'model_uri': 'models:/m-fe5abeb5f72d417284b8ee7a3300235d',
 'train_metrics': {'accuracy': 0.8938388625592417,
  'precision': 0.8176470588235294,
  'recall': 0.4186746987951807,
  'auc': 0.9314479515361784},
 'valid_metrics': {'accuracy': 0.8923076923076924,
  'precision': 0.8148148148148148,
  'recall': 0.4049079754601227,
  'auc': 0.9208574966247176}}

### Decision Tree Classifier

In [15]:
from sklearn.tree import DecisionTreeClassifier

simple_tree = DecisionTreeClassifier(
    criterion="gini", splitter="best", max_depth=10, min_samples_split=2, min_samples_leaf=1
)
track_experiment(df, simple_tree, "simple_tree")

🏃 View run simple_tree at: http://127.0.0.1:5000/#/experiments/2/runs/cc447aa0af7144d0a2fb0df750e552f2
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


{'run_id': 'cc447aa0af7144d0a2fb0df750e552f2',
 'model_uri': 'models:/m-bc0c58b5f86b496185506951344a1cf5',
 'train_metrics': {'accuracy': 0.9796208530805687,
  'precision': 0.9737704918032787,
  'recall': 0.8945783132530121,
  'auc': 0.9964763440714489},
 'valid_metrics': {'accuracy': 0.9384615384615385,
  'precision': 0.8278145695364238,
  'recall': 0.7668711656441718,
  'auc': 0.9405215773236983}}

### Random Forecast Classifier

In [16]:
from sklearn.ensemble import RandomForestClassifier

random_forest = RandomForestClassifier(
    n_estimators=200, criterion="gini", max_depth=10, min_samples_split=2, min_samples_leaf=1, bootstrap=True
)
track_experiment(df, random_forest, "random_forest")

🏃 View run random_forest at: http://127.0.0.1:5000/#/experiments/2/runs/7c3e4ede73f045db964ad33699a711b3
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


{'run_id': '7c3e4ede73f045db964ad33699a711b3',
 'model_uri': 'models:/m-3fcc9045f45a4facb0f3b62395c7d57b',
 'train_metrics': {'accuracy': 0.9796208530805687,
  'precision': 0.933933933933934,
  'recall': 0.9367469879518072,
  'auc': 0.9980823180235},
 'valid_metrics': {'accuracy': 0.9442307692307692,
  'precision': 0.8431372549019608,
  'recall': 0.7914110429447853,
  'auc': 0.9819063875034102}}

### Gradient Boosting Classifier

In [17]:
from sklearn.ensemble import GradientBoostingClassifier

grad_boost = GradientBoostingClassifier(
    n_estimators=200,
    loss="log_loss",
    learning_rate=0.1,
    random_state=42,
    max_depth=10,
    min_samples_split=2,
    min_samples_leaf=1,
)
track_experiment(df, grad_boost, "gradient_boosting")

C:\Users\amade\MLOps_amadeov3\.venv\Lib\site-packages\sklearn\ensemble\_gb.py:666: FutureWarning: The parameter `criterion` is deprecated and will be removed in 1.11. It has no effect. Leave it to its default value to avoid this warning.
  warnings.warn(


🏃 View run gradient_boosting at: http://127.0.0.1:5000/#/experiments/2/runs/fb4d90d9dd6c453d94aa7499bbfec91c
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


{'run_id': 'fb4d90d9dd6c453d94aa7499bbfec91c',
 'model_uri': 'models:/m-a7e4266a6fa546469b61bd083d9629e6',
 'train_metrics': {'accuracy': 0.9924170616113744,
  'precision': 0.9759036144578314,
  'recall': 0.9759036144578314,
  'auc': 0.9997475842628105},
 'valid_metrics': {'accuracy': 0.9538461538461539,
  'precision': 0.8758169934640523,
  'recall': 0.8220858895705522,
  'auc': 0.9818259403571854}}

### XGBoost

In [22]:
from xgboost import XGBClassifier

xgb = XGBClassifier(
    objective="binary:logistic", n_estimators=500, learning_rate=0.05, max_depth=10, reg_alpha=0.1, random_state=42
)
track_experiment(df, xgb, "xgboost")

2026/10/19 14:08:48 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run xgboost at: http://127.0.0.1:5000/#/experiments/2/runs/7556fe66451d450c9dfcf0162fe3c56e
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


{'run_id': '7556fe66451d450c9dfcf0162fe3c56e',
 'model_uri': 'models:/m-28d37c87224144c58c3aa632dda873c7',
 'train_metrics': {'accuracy': 0.9924170616113744,
  'precision': 0.9759036144578314,
  'recall': 0.9759036144578314,
  'auc': 0.999718785151856},
 'valid_metrics': {'accuracy': 0.9490384615384615,
  'precision': 0.8313253012048193,
  'recall': 0.8466257668711656,
  'auc': 0.9848864296157425}}

### LightGBM

In [23]:
from lightgbm import LGBMClassifier

lgb = LGBMClassifier(
    objective="binary",
    metric="auc",
    n_estimators=500,
    learning_rate=0.1,
    num_leaves=5,
    min_child_samples=15,
    reg_lambda=1.0,
    random_state=42,
    verbosity=-1,
)
track_experiment(df, lgb, "lightgbm")

2026/10/19 14:08:58 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run lightgbm at: http://127.0.0.1:5000/#/experiments/2/runs/62f4cce712384739a78a6e3e5e26a3e2
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


{'run_id': '62f4cce712384739a78a6e3e5e26a3e2',
 'model_uri': 'models:/m-9f24f420806c464997cd23870e4bc0af',
 'train_metrics': {'accuracy': 0.9843601895734597,
  'precision': 0.930835734870317,
  'recall': 0.9728915662650602,
  'auc': 0.9982923821269329},
 'valid_metrics': {'accuracy': 0.9538461538461539,
  'precision': 0.8285714285714286,
  'recall': 0.8895705521472392,
  'auc': 0.9850193422921141}}

### Small Neural Network

In [24]:
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

nn = make_pipeline(
    StandardScaler(),
    MLPClassifier(
        hidden_layer_sizes=(64, 32, 16),
        activation="tanh",
        solver="adam",
        alpha=0.1,
        learning_rate_init=0.03,
        max_iter=1000,
        early_stopping=True,
        validation_fraction=0.15,
        n_iter_no_change=20,
        random_state=42,
    ),
)
track_experiment(df, nn, "neural_network")

2026/10/19 14:09:10 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run neural_network at: http://127.0.0.1:5000/#/experiments/2/runs/524eb5f137bd4b8883c10e9ccac68eb7
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


{'run_id': '524eb5f137bd4b8883c10e9ccac68eb7',
 'model_uri': 'models:/m-1fa88f3367c247768769910a28633a5b',
 'train_metrics': {'accuracy': 0.9559241706161138,
  'precision': 0.8169761273209549,
  'recall': 0.927710843373494,
  'auc': 0.9882381042731105},
 'valid_metrics': {'accuracy': 0.9528846153846153,
  'precision': 0.8275862068965517,
  'recall': 0.8834355828220859,
  'auc': 0.9787864373106869}}

## Recover Model from Model Registry

In [ ]:
# Let's recover from MLflow the best performing model:
lightgbm_uri = "models:/LightGBM /1"

lgb = mlflow.pyfunc.load_model(lightgbm_uri)
lgb.predict(df.drop(columns=["churn"]))

array([0, 0, 0, ..., 0, 0, 1], shape=(3150,))